In [1]:
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

from config import PINECONE_KEY

/opt/conda/envs/pine/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
files = pd.read_csv("files/course_descriptions.csv", encoding="cp1252")
files

,course_name,course_slug,course_technology,course_description,course_topic,course_description_short
0,Introduction to Tableau,tableau,tableau,Tableau is now one of the most popular busines...,data visualization,Teaching you how to tell compelling stories wi...
1,The Complete Data Visualization Course with Py...,data-visualization,python,The Data Visualization course is designed for ...,data visualization,Teaching you how to master the art of creating...
2,Introduction to R Programming,introduction-to-r-programming,r,R is one of the best programming languages spe...,programming,"Providing you with the skills to manipulate, a..."
3,Data Preprocessing with NumPy,data-preprocessing-numpy,python,This course is designed to show you how to wor...,data processing,This course will guide you through one of Pyth...
4,Introduction to Data and Data Science,intro-to-data-and-data-science,theory,Working with data is an essential part of main...,machine learning,Introducing you to the field of data science a...
...,...,...,...,...,...,...
101,Intro to NLP for AI,intro-to-nlp-for-ai,python,Natural language processing is an exciting and...,programming,Unlock the power of natural language processin...
102,Data Analysis with ChatGPT,data-analysis-with-chatgpt,chatgpt,Leverage ChatGPT's Advanced Data Analysis Code...,programming,Master ChatGPT for data analysis. Boost your p...
103,ChatGPT for Data Science,chatgpt-for-data-science,chatgpt,"In this course, you will learn how to utilize ...",machine learning,Learn how to increase your productivity using ...
104,Intro to LLMs,intro-to-llms,python,"In recent years, large language models (LLMs) ...",machine learning,This LLM course will guide you step-by-step th...


In [6]:
def create_course_description(row):
    return f"The course name is {row['course_name']}, the slug is {row['course_slug']}, the technology is {row['course_technology']}, and the course topic is {row['course_topic']}."

# print(create_course_description(files.iloc[0]))
files["course_info"] = files.apply(create_course_description, axis=1)  # should apply the function to each row and save a new column of data.

### Embedding the data

In [2]:
pc = Pinecone(api_key=PINECONE_KEY)
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:03<00:00, 33.45it/s]


In [17]:
def create_embeddings(row):
  combined_text = " ".join([str(row[field]) for field in ["course_description", "course_info", "course_description_short"]])
  embedding = model.encode(combined_text)
  return embedding

files["embedding"] = files.apply(create_embeddings, axis=1)

In [19]:
# Create vectors by extracting the course name as an ID and the embedding vector for each row in the DataFrame, preparing them for upserting into a vector database.
vectors_to_upsert = [(str(row["course_name"]), row["embedding"].tolist()) for _, row in files.iterrows()]

In [23]:
pc.create_index(
    name="third",
    dimension=model.get_sentence_embedding_dimension(),
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

/tmp/ipykernel_1168/3753715217.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension=model.get_sentence_embedding_dimension(),


Name:,third
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://third-7mk7crn.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [4]:
third = pc.index("third")

In [ ]:
third.upsert(vectors=vectors_to_upsert)
print("Done!")

Done!


### Embedding Search

In [5]:
query = "clustering"
query_embedding = model.encode(query, show_progress_bar=False).tolist()
query_results = third.query(vector=[query_embedding], top_k=12, include_values=False)

In [6]:
for item in query_results.matches:
  print(f"Topic: {item.id}, Score: {item.score}")

Topic: Machine Learning in Excel, Score: 0.361535102
Topic: Machine Learning with K-Nearest Neighbors, Score: 0.316150218
Topic: Customer Churn Analysis with SQL and Tableau, Score: 0.276414901
Topic: Machine Learning in Python, Score: 0.265437156
Topic: Growth Analysis with SQL, Python, and Tableau  , Score: 0.256822616
Topic: Linear Algebra and Feature Selection, Score: 0.253460914
Topic: Customer Engagement Analysis with SQL and Tableau, Score: 0.232827201
Topic: Fashion Analytics with Tableau, Score: 0.232393295
Topic: Machine Learning with Support Vector Machines, Score: 0.226737991
Topic: Machine Learning with Naive Bayes, Score: 0.225633159
Topic: Data Analysis with Excel Pivot Tables, Score: 0.220349327
Topic: Data Preprocessing with NumPy, Score: 0.217773467


### Data preprocessing and embedding for courses with section data

In [8]:
files = pd.read_csv("files/course_section_descriptions.csv", encoding="cp1252")
files["unique_id"] = files["course_id"].astype(str) + "-" + files["section_id"].astype(str)
files["metadata"] = files.apply(lambda row: {
    "course_name": row["course_name"],
    "section_name": row["section_name"],
    "section_description": row["section_description"]
}, axis=1)
files

,course_id,course_name,course_slug,course_description,course_description_short,course_technology,course_topic,course_instructor_quote,section_id,section_name,section_description,unique_id,metadata
0,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,9,Introduction to Tableau,While Tableau is an indispensable tool in the ...,2-9,"{'course_name': 'Introduction to Tableau', 'se..."
1,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,10,Tableau Functionalities,"In this section, you will create your first Ta...",2-10,"{'course_name': 'Introduction to Tableau', 'se..."
2,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,11,The Tableau Exercise,This section is a practical example that will ...,2-11,"{'course_name': 'Introduction to Tableau', 'se..."
3,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,12,Introduction,"In this section, you will learn about the impo...",3-12,{'course_name': 'The Complete Data Visualizati...
4,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,13,Setting Up the Environments,"Here, we set up different environments for the...",3-13,{'course_name': 'The Complete Data Visualizati...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
675,117,"Growth Analysis with SQL, Python, and Tableau",growth-analysis-with-sql-python-and-tableau,Every company aims to maximize its market pres...,Master the creation of a growth data dashboard...,tableau,data visualization,NaN,819,Case Study Overview,This part introduces the terminology and highl...,117-819,"{'course_name': 'Growth Analysis with SQL, Pyt..."
676,117,"Growth Analysis with SQL, Python, and Tableau",growth-analysis-with-sql-python-and-tableau,Every company aims to maximize its market pres...,Master the creation of a growth data dashboard...,tableau,data visualization,NaN,820,Retrieving Relevant Data from the Database,"Here, we explore the 365 database utilized in ...",117-820,"{'course_name': 'Growth Analysis with SQL, Pyt..."
677,117,"Growth Analysis with SQL, Python, and Tableau",growth-analysis-with-sql-python-and-tableau,Every company aims to maximize its market pres...,Master the creation of a growth data dashboard...,tableau,data visualization,NaN,821,Crafting the Graphs,"Once all data sources are ready, we'll start c...",117-821,"{'course_name': 'Growth Analysis with SQL, Pyt..."
678,117,"Growth Analysis with SQL, Python, and Tableau",growth-analysis-with-sql-python-and-tableau,Every company aims to maximize its market pres...,Master the creation of a growth data dashboard...,tableau,data visualization,NaN,822,Assembling the Dashboard,We'll organize all completed graphs across var...,117-822,"{'course_name': 'Growth Analysis with SQL, Pyt..."


In [10]:
def create_embeddings(row):
  combined_text = " ".join([str(row[field]) for field in ["course_name", "course_technology", "course_description", "section_name", "section_description"]])
  embedding = model.encode(combined_text)
  return embedding

In [12]:
files["embedding"] = files.apply(create_embeddings, axis=1)

In [14]:
files.shape

(680, 14)

In [17]:
vectors = [(row["unique_id"], row["embedding"].tolist(), row["metadata"]) for _, row in files.iterrows()]

In [ ]:
pc.create_index(
    name="second",
    dimension=model.get_embedding_dimension(),
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)
second = pc.index("second")
second.upsert(vectors=vectors)